In [4]:
# 1. Cargar datos
import pandas as pd
df = pd.read_csv(r"C:\Users\Ing. Antonio Rial\OneDrive - Universidad Austral\MCD_Laboratorio.de.Implementación.2\Competencia.House.Pricing\data\tabular\train_processed.csv", low_memory=False)

In [ ]:
import os

def verificar_y_asegurar_minusculas(ruta_archivo):
    # 1. Cargar el CSV
    try:
        # Intentamos utf-8, si falla usamos latin-1 (común en datasets reales)
        try:
            df = pd.read_csv(ruta_archivo, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(ruta_archivo, encoding='latin-1')
            
        print(f"✅ Archivo cargado exitosamente: {df.shape[0]} filas x {df.shape[1]} columnas")
    except Exception as e:
        print(f"❌ Error al leer el archivo: {e}")
        return

    # 2. Identificar columnas de tipo texto (object)
    cols_texto = df.select_dtypes(include=['object']).columns
    if len(cols_texto) == 0:
        print("ℹ️ No se encontraron columnas de tipo texto en el dataset.")
        return

    print(f"\n🔍 Analizando {len(cols_texto)} columnas de texto...\n")

    # 3. Verificar consistencia
    inconsistencias = {}
    for col in cols_texto:
        valores_validos = df[col].dropna()
        if valores_validos.empty:
            continue

        # Detectar valores que NO están completamente en minúsculas
        mascara_mayusculas = valores_validos != valores_validos.str.lower()

        if mascara_mayusculas.any():
            cantidad = mascara_mayusculas.sum()
            ejemplos = valores_validos[mascara_mayusculas].unique()[:3].tolist()
            inconsistencias[col] = {'cantidad': cantidad, 'ejemplos': ejemplos}

    # 4. Reporte de resultados
    if not inconsistencias:
        print("✅ ¡Perfecto! Todas las columnas de texto ya están consistentemente en minúsculas.")
    else:
        print("⚠️  Se encontraron inconsistencias (mayúsculas o mezcla de casos):")
        for col, info in inconsistencias.items():
            print(f"   📂 '{col}': {info['cantidad']} valores. Ej: {info['ejemplos']}")

    # 5. Asegurar minúsculas (conversión automática)
    print("\n🔄 Convirtiendo automáticamente todo el texto a minúsculas...")
    for col in cols_texto:
        # pandas str.lower() ignora correctamente los NaN
        df[col] = df[col].str.lower()

    # 6. Guardar archivo corregido
    base, ext = os.path.splitext(ruta_archivo)
    ruta_salida = f"{base}_consistente{ext}"
    df.to_csv(ruta_salida, index=False, encoding='utf-8')
    print(f"✅ Archivo corregido guardado como: {ruta_salida}")

if __name__ == "__main__":
    # 👉 Cambia aquí el nombre/ruta de tu archivo si es necesario
    verificar_y_asegurar_minusculas(r"C:\Users\Ing. Antonio Rial\OneDrive - Universidad Austral\MCD_Laboratorio.de.Implementación.2\Competencia.House.Pricing\data\tabular\train_processed.csv")

✅ Archivo cargado exitosamente: 11840 filas x 46 columnas

🔍 Analizando 2 columnas de texto...

⚠️  Se encontraron inconsistencias (mayúsculas o mezcla de casos):
   📂 'description': 11818 valores. Ej: ['This 665 square foot condo home has 1 bedrooms and 1.0 bathrooms. This home is located at 511 Lucerne Ave APT 314, Lake Worth, FL 33460.', 'This 1030 square foot condo home has 2 bedrooms and 2.0 bathrooms. This home is located at 7601 E Treasure Dr APT 1603, North Bay Village, FL 33141.', 'This 980 square foot condo home has 2 bedrooms and 2.0 bathrooms. This home is located at 750 SE 6th Ave APT 232, Deerfield Beach, FL 33441.']
   📂 'homeType': 11840 valores. Ej: ['CONDO', 'SINGLE_FAMILY', 'APARTMENT']

🔄 Convirtiendo automáticamente todo el texto a minúsculas...
✅ Archivo corregido guardado como: C:\Users\Ing. Antonio Rial\OneDrive - Universidad Austral\MCD_Laboratorio.de.Implementación.2\Competencia.House.Pricing\data\tabular\train_processed_consistente.csv
